# Statistical Rigor Evaluation of phi-4 Arrhenius Results

This notebook reproduces the statistical rigor evaluation from `eval.py` (artifact `art_lriKx3K3rntq`),
which addresses six reviewer critiques of the phi-4 Arrhenius inference-energy experiment.

**What it does:**
- Loads 30 OC (Overconfidence-at-threshold) MMLU-Pro instances from the parent experiment
- Computes six statistical metrics (McNemar, Clopper-Pearson CIs, bootstrap R², subgroup analysis, power, T_TURN reconciliation)
- All computation is pure Python/numpy/scipy — zero API calls, zero cost

**Reviewer critiques addressed:**
1. McNemar's exact test on paired BON outcomes
2. Clopper-Pearson + Wilson CIs on Table 2 accuracy rows
3. Bootstrap CI on median R²
4. T_pref=1.0 concentration subgroup analysis
5. Power analysis reframing for Ea–T_pref correlation
6. T_TURN^theory vs T_TURN^emp numerical reconciliation

In [1]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru — NOT pre-installed on Colab, always install
_pip('loguru==0.7.3')

# Core packages — pre-installed on Colab, install locally only to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'scipy==1.16.3', 'matplotlib==3.10.0')


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import json
import math
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from loguru import logger
from scipy import stats


class _NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.bool_):
            return bool(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

1

In [3]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-ce2af9-arrhenius-kinetics-of-llm-inference-acti/main/round-4/evaluation-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [4]:
data = load_data()
print(f"Loaded data with {len(data['datasets'][0]['examples'])} examples")

Loaded data with 30 examples


In [5]:
# ── Config ── Tune these to trade off speed vs thoroughness ──────────────────
N_BON = 16          # BON samples per instance (matches original experiment)
N_CHOICES = 10      # MMLU-Pro has 10 answer choices (K)
TEMP_GRID = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 1.0]

# Bootstrap resamples: original=10000, minimum=100
B = 100             # original: 10000 — increase for stable CI estimates
BOOTSTRAP_SEED = 42

# Derived constant
BON_THRESHOLD = 1 - 0.5 ** (1.0 / N_BON)  # ~0.0416: p > this → P_BON > 0.5
print(f"BON_THRESHOLD = {BON_THRESHOLD:.4f}")
print(f"Bootstrap resamples B = {B}")

BON_THRESHOLD = 0.0424
Bootstrap resamples B = 100


## Utility Functions

Three shared helpers used throughout:
- `wilson_ci`: Wilson score confidence interval for proportions
- `clopper_pearson_ci`: Exact binomial CI (more conservative, used in Table 2)
- `interp_p_correct`: Linear interpolation of P(correct|T) between temperature grid points
- `bon_binary`: Converts P(correct) to expected BON outcome (True/False)

In [6]:
def wilson_ci(k: int, n: int, alpha: float = 0.05) -> tuple:
    """Wilson score interval."""
    z = stats.norm.ppf(1 - alpha / 2)
    denom = n + z ** 2
    center = (k + z ** 2 / 2) / denom
    margin = z * math.sqrt(k * (n - k) / n + z ** 2 / 4) / denom
    return max(0.0, center - margin), min(1.0, center + margin)


def clopper_pearson_ci(k: int, n: int, alpha: float = 0.05) -> tuple:
    """Exact Clopper-Pearson binomial CI."""
    lo = stats.beta.ppf(alpha / 2, k, n - k + 1) if k > 0 else 0.0
    hi = stats.beta.ppf(1 - alpha / 2, k + 1, n - k) if k < n else 1.0
    return lo, hi


def interp_p_correct(p_by_t: dict, temp: float) -> float:
    """Return P(correct) at `temp` by linear interpolation between grid points."""
    temps = sorted(p_by_t.keys())
    vals = [p_by_t[t] for t in temps]
    if temp <= temps[0]:
        return vals[0]
    if temp >= temps[-1]:
        return vals[-1]
    for i in range(len(temps) - 1):
        t0, t1 = temps[i], temps[i + 1]
        if t0 <= temp <= t1:
            frac = (temp - t0) / (t1 - t0)
            return vals[i] + frac * (vals[i + 1] - vals[i])
    return vals[-1]


def bon_binary(p_correct: float, n: int = N_BON) -> bool:
    """Expected BON correct if P(correct|T) > threshold."""
    return p_correct > BON_THRESHOLD

## Load OC Instances

The evaluation focuses on "OC" (Overconfidence-at-threshold) instances — MMLU-Pro questions where
the Arrhenius model identifies a temperature threshold above which the model becomes correct.

These are filtered by `predict_is_oc_error == "true"` from the parent experiment's output.

In [7]:
# Load OC instances from the already-loaded data dict
examples = data["datasets"][0]["examples"]
oc_instances = [e for e in examples if str(e.get("predict_is_oc_error", "")).lower() == "true"]
metadata = data["metadata"]

logger.info(f"Total examples: {len(examples)}")
logger.info(f"OC instances: {len(oc_instances)}")
print(f"\nFirst OC instance fields: {list(oc_instances[0].keys())}")

20:53:31|INFO   |Total examples: 30


20:53:31|INFO   |OC instances: 30



First OC instance fields: ['input', 'output', 'metadata_question_id', 'metadata_subject', 'metadata_split', 'metadata_num_choices', 'metadata_src', 'predict_is_oc_error', 'predict_arrhenius_Ea', 'predict_arrhenius_R2', 'predict_arrhenius_log_A', 'predict_Delta', 'predict_T_pref', 'predict_T_TURN', 'predict_T_thresh_N16_simplified', 'predict_T_thresh_N16_approx', 'predict_T_thresh_N16_A_corrected', 'predict_p_correct_by_T', 'predict_R2_advantage_vs_alternatives', 'predict_R2_linear', 'predict_R2_exp_T', 'predict_R2_power_law', 'predict_T_thresh_N16_empirical_min', 'predict_verdict']


## Metric 1: McNemar's Exact Test

**Reviewer critique:** The paper claims T_operating beats fixed-temperature strategies but provides
no significance test on paired outcomes.

**Method:** Reconstruct per-instance BON (Best-of-N) correctness outcomes from
`predict_p_correct_by_T` using P_BON > 0.5 threshold (N=16). Build 2×2 contingency tables
and apply McNemar's exact test (two-sided binomial) to compare:
- T_operating (+0.3 above threshold) vs fixed T=0.7
- T_operating (+0.3 above threshold) vs fixed T=1.0

In [8]:
def compute_mcnemar(oc_instances: list) -> dict:
    logger.info("=== Metric 1: McNemar's exact test ===")
    outcomes = []
    for e in oc_instances:
        p_by_t_raw = e["predict_p_correct_by_T"]
        if isinstance(p_by_t_raw, str):
            p_by_t_raw = json.loads(p_by_t_raw)
        p_by_t = {float(k): float(v) for k, v in p_by_t_raw.items()}

        t_thresh = float(e["predict_T_thresh_N16_simplified"])
        t_op_03 = t_thresh + 0.3

        p_op03 = interp_p_correct(p_by_t, t_op_03)
        p_t07 = interp_p_correct(p_by_t, 0.7)
        p_t10 = interp_p_correct(p_by_t, 1.0)

        outcomes.append({
            "bon_Toperating_03": bon_binary(p_op03),
            "bon_fixed07": bon_binary(p_t07),
            "bon_fixed10": bon_binary(p_t10),
            "p_op03": p_op03,
            "p_t07": p_t07,
            "p_t10": p_t10,
        })

    n = len(outcomes)
    # Verify reconstruction vs reported aggregate
    k_op03 = sum(o["bon_Toperating_03"] for o in outcomes)
    k_t07 = sum(o["bon_fixed07"] for o in outcomes)
    k_t10 = sum(o["bon_fixed10"] for o in outcomes)
    logger.info(f"Reconstructed accuracy: T_op_0.3={k_op03}/{n}={k_op03/n:.3f} (reported 0.900)")
    logger.info(f"Reconstructed accuracy: fixed_T07={k_t07}/{n}={k_t07/n:.3f} (reported 0.833)")
    logger.info(f"Reconstructed accuracy: fixed_T10={k_t10}/{n}={k_t10/n:.3f} (reported 0.933)")

    # 2×2 contingency tables
    def build_table(o1_key, o2_key):
        a = sum(o[o1_key] and o[o2_key] for o in outcomes)       # both correct
        b = sum(o[o1_key] and not o[o2_key] for o in outcomes)   # o1 only
        c = sum(not o[o1_key] and o[o2_key] for o in outcomes)   # o2 only
        d = sum(not o[o1_key] and not o[o2_key] for o in outcomes)  # both wrong
        return a, b, c, d

    def mcnemar_exact(b, c):
        """McNemar's exact test (two-sided) using binomial distribution."""
        n_disc = b + c
        if n_disc == 0:
            return 1.0
        # Two-sided: 2 * P(X <= min(b,c)) under Binomial(n_disc, 0.5)
        p = 2.0 * float(stats.binom.cdf(min(b, c), n_disc, 0.5))
        return min(p, 1.0)

    # T_operating vs fixed_T07
    a1, b1, c1, d1 = build_table("bon_Toperating_03", "bon_fixed07")
    p1 = mcnemar_exact(b1, c1)
    logger.info(f"McNemar T_op_0.3 vs fixed_T07: table={[a1,b1,c1,d1]}, p={p1:.4f}")

    # T_operating vs fixed_T10
    a2, b2, c2, d2 = build_table("bon_Toperating_03", "bon_fixed10")
    p2 = mcnemar_exact(b2, c2)
    logger.info(f"McNemar T_op_0.3 vs fixed_T10: table={[a2,b2,c2,d2]}, p={p2:.4f}")

    caveat = (
        f"The 6.7 pp difference ({abs(b1 - c1)} instance(s) net advantage out of {n}) "
        f"is directionally consistent with the Arrhenius prediction but does not reach statistical "
        f"significance (McNemar p={p1:.3f}, n={n}); confirming an effect of this size at 80% "
        f"power requires approximately n≈200 valid instances."
    )

    return {
        "mcnemar_Toperating_vs_fixed07": {
            "pvalue": round(p1, 6),
            "contingency_table": [a1, b1, c1, d1],
            "description": "a=both_correct, b=Toperating_only, c=fixed07_only, d=both_wrong",
        },
        "mcnemar_Toperating_vs_fixed10": {
            "pvalue": round(p2, 6),
            "contingency_table": [a2, b2, c2, d2],
            "description": "a=both_correct, b=Toperating_only, c=fixed10_only, d=both_wrong",
        },
        "reconstruction_note": (
            "Per-instance BON outcomes reconstructed from predict_p_correct_by_T using "
            f"BON threshold p>{BON_THRESHOLD:.4f} (P_BON>0.5 with N={N_BON}). "
            f"Reconstructed k: T_op_0.3={k_op03}, fixed_T07={k_t07}, fixed_T10={k_t10} "
            f"(reported: 27, 25, 28 respectively)."
        ),
        "mandatory_caveat": caveat,
        "per_instance_outcomes": outcomes,  # stored for subgroup analysis reuse
    }


mcnemar_res = compute_mcnemar(oc_instances)

20:53:31|INFO   |=== Metric 1: McNemar's exact test ===


20:53:31|INFO   |Reconstructed accuracy: T_op_0.3=26/30=0.867 (reported 0.900)


20:53:31|INFO   |Reconstructed accuracy: fixed_T07=26/30=0.867 (reported 0.833)


20:53:31|INFO   |Reconstructed accuracy: fixed_T10=28/30=0.933 (reported 0.933)


20:53:31|INFO   |McNemar T_op_0.3 vs fixed_T07: table=[24, 2, 2, 2], p=1.0000


20:53:31|INFO   |McNemar T_op_0.3 vs fixed_T10: table=[26, 0, 2, 2], p=0.5000


## Metric 2: Clopper-Pearson + Wilson CIs on Table 2

**Reviewer critique:** Table 2 reports point estimates without confidence intervals,
making it impossible to judge whether strategy differences are statistically meaningful.

**Method:** Compute exact Clopper-Pearson and Wilson score 95% CIs for all 6 strategies
at n=30. Check whether each strategy's CI overlaps with the reference (T_operating_delta_0.3).

**Expected finding:** All CIs overlap → strategy differences are statistically indistinguishable at n=30.

In [9]:
def compute_table2_cis(metadata: dict) -> tuple:
    logger.info("=== Metric 2: Clopper-Pearson + Wilson CIs ===")
    acc_map = metadata["step7_accuracy_comparison"]["accuracy"]
    n = int(acc_map["n_instances"])

    strategies = [
        ("T_operating_delta_0.2", acc_map["T_operating_delta_0.2"]),
        ("T_operating_delta_0.3", acc_map["T_operating_delta_0.3"]),
        ("T_operating_delta_0.4", acc_map["T_operating_delta_0.4"]),
        ("fixed_T07",              acc_map["fixed_T07"]),
        ("TURN_adapted",           acc_map["TURN_adapted"]),
        ("fixed_T10",              acc_map["fixed_T10"]),
    ]

    ref_T_op_03_acc = acc_map["T_operating_delta_0.3"]
    ref_T_op_03_k = round(ref_T_op_03_acc * n)

    rows = []
    for name, acc in strategies:
        k = round(acc * n)
        cp_lo, cp_hi = clopper_pearson_ci(k, n)
        w_lo, w_hi = wilson_ci(k, n)
        ref_cp_lo, ref_cp_hi = clopper_pearson_ci(ref_T_op_03_k, n)
        overlaps = (cp_lo <= ref_cp_hi) and (ref_cp_lo <= cp_hi)
        rows.append({
            "strategy": name,
            "k": k,
            "n": n,
            "accuracy": round(acc, 6),
            "cp_lo": round(cp_lo, 4),
            "cp_hi": round(cp_hi, 4),
            "wilson_lo": round(w_lo, 4),
            "wilson_hi": round(w_hi, 4),
            "cis_overlap_T_operating_0.3": overlaps,
        })
        logger.info(f"  {name}: {k}/{n}={acc:.3f} CP=[{cp_lo:.3f},{cp_hi:.3f}] W=[{w_lo:.3f},{w_hi:.3f}] overlap={overlaps}")

    n_overlap = sum(1 for r in rows if r["cis_overlap_T_operating_0.3"] and r["strategy"] != "T_operating_delta_0.3")
    logger.info(f"{n_overlap}/{len(rows)-1} strategies have overlapping CIs with T_operating_delta_0.3")

    return rows, {"n_strategies_ci_overlap_with_top": n_overlap}


def plot_table2_cis(rows: list, out_path=None):
    logger.info("Plotting table2_with_CIs")
    sorted_rows = sorted(rows, key=lambda r: r["accuracy"], reverse=True)
    names = [r["strategy"] for r in sorted_rows]
    accs = [r["accuracy"] for r in sorted_rows]
    lo_err = [r["accuracy"] - r["cp_lo"] for r in sorted_rows]
    hi_err = [r["cp_hi"] - r["accuracy"] for r in sorted_rows]
    ks = [r["k"] for r in sorted_rows]
    ns = [r["n"] for r in sorted_rows]
    cp_los = [r["cp_lo"] for r in sorted_rows]
    cp_his = [r["cp_hi"] for r in sorted_rows]

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = range(len(names))
    bars = ax.barh(list(y_pos), accs, xerr=[lo_err, hi_err],
                   capsize=5, color="#4C9BE8", alpha=0.75, error_kw={"ecolor": "#1a1a2e", "linewidth": 1.5})
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(names, fontsize=10)
    ax.set_xlabel("BON Accuracy (N=16)", fontsize=11)
    ax.set_title(
        "Table 2: BON Accuracy with 95% Clopper-Pearson CIs (n=30)\n"
        "Note: All CI pairs overlap — strategy differences are statistically indistinguishable",
        fontsize=11,
    )
    ax.axvline(0.833, color="#E84C4C", linestyle="--", linewidth=1.2, label="fixed_T07 (0.833)")
    ax.axvline(0.900, color="#4CE869", linestyle="--", linewidth=1.2, label="T_operating_0.3 (0.900)")
    ax.set_xlim(0.55, 1.10)

    for i, (k, n, lo, hi) in enumerate(zip(ks, ns, cp_los, cp_his)):
        ax.text(1.01, i, f"{k}/{n} [CI: {lo:.2f}\u2013{hi:.2f}]", va="center", fontsize=8.5)

    ax.legend(fontsize=9)
    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        logger.info(f"Saved {out_path}")
    plt.show()
    plt.close(fig)


table2_cis, table2_ci_agg = compute_table2_cis(metadata)
plot_table2_cis(table2_cis, out_path="table2_with_CIs.png")

20:53:31|INFO   |=== Metric 2: Clopper-Pearson + Wilson CIs ===


20:53:31|INFO   |  T_operating_delta_0.2: 26/30=0.867 CP=[0.693,0.962] W=[0.703,0.947] overlap=True


20:53:31|INFO   |  T_operating_delta_0.3: 27/30=0.900 CP=[0.735,0.979] W=[0.744,0.965] overlap=True


20:53:31|INFO   |  T_operating_delta_0.4: 24/30=0.800 CP=[0.614,0.923] W=[0.627,0.905] overlap=True


20:53:31|INFO   |  fixed_T07: 25/30=0.833 CP=[0.653,0.944] W=[0.664,0.927] overlap=True


20:53:31|INFO   |  TURN_adapted: 29/30=0.967 CP=[0.828,0.999] W=[0.833,0.994] overlap=True


20:53:31|INFO   |  fixed_T10: 28/30=0.933 CP=[0.779,0.992] W=[0.787,0.982] overlap=True


20:53:31|INFO   |5/5 strategies have overlapping CIs with T_operating_delta_0.3


20:53:31|INFO   |Plotting table2_with_CIs


20:53:31|INFO   |Saved table2_with_CIs.png


## Metric 3: Bootstrap CI on Median R²

**Reviewer critique:** The paper reports median R²=0.85 and claims the C1 criterion is met,
but does not quantify estimation uncertainty around the median.

**Method:** B bootstrap resamples (with replacement, seed=42) of the 30 R² values.
Check whether the C1 threshold of 0.85 falls within the 95% bootstrap CI.

**Expected finding:** The 0.85 threshold falls within the CI → verdict is INDETERMINATE.

In [10]:
def compute_bootstrap_r2(oc_instances: list) -> dict:
    logger.info("=== Metric 3: Bootstrap CI on median R² ===")
    r2_vals = np.array([float(e["predict_arrhenius_R2"]) for e in oc_instances])
    n = len(r2_vals)
    obs_median = float(np.median(r2_vals))
    logger.info(f"Observed median R²={obs_median:.6f} from n={n} instances")

    rng = np.random.default_rng(BOOTSTRAP_SEED)
    boot_medians = np.array([
        np.median(rng.choice(r2_vals, size=n, replace=True))
        for _ in range(B)
    ])
    ci_lo = float(np.percentile(boot_medians, 2.5))
    ci_hi = float(np.percentile(boot_medians, 97.5))
    logger.info(f"Bootstrap 95% CI on median R²: [{ci_lo:.4f}, {ci_hi:.4f}]")

    threshold_in_ci = ci_lo <= 0.85 <= ci_hi
    interp_text = (
        f"Median R²={obs_median:.3f} (95% bootstrap CI: [{ci_lo:.3f}, {ci_hi:.3f}]); "
        f"the C1 threshold of 0.85 is {abs(0.85-obs_median):.3f} above the observed median, "
        f"which {'falls within' if threshold_in_ci else 'falls outside'} bootstrap estimation error — "
        "the binary met/not-met judgment is statistically indeterminate."
    )
    logger.info(interp_text)

    return {
        "median": round(obs_median, 6),
        "ci_lo": round(ci_lo, 6),
        "ci_hi": round(ci_hi, 6),
        "n_resamples": B,
        "threshold_085_in_ci": threshold_in_ci,
        "interpretation": interp_text,
    }


bootstrap_res = compute_bootstrap_r2(oc_instances)

20:53:31|INFO   |=== Metric 3: Bootstrap CI on median R² ===


20:53:31|INFO   |Observed median R²=0.847721 from n=30 instances


20:53:31|INFO   |Bootstrap 95% CI on median R²: [0.7705, 0.9111]


20:53:31|INFO   |Median R²=0.848 (95% bootstrap CI: [0.771, 0.911]); the C1 threshold of 0.85 is 0.002 above the observed median, which falls within bootstrap estimation error — the binary met/not-met judgment is statistically indeterminate.


## Metric 4: T_pref=1.0 Subgroup Analysis

**Reviewer critique:** 43% of OC instances have T_pref=1.0 (ceiling). When T_pref=1.0,
T_operating (thresh + 0.3) may coincide with fixed T=1.0, making the comparison degenerate.

**Method:** Split OC instances into T_pref=1.0 vs T_pref<1.0 subgroups.
Compute subgroup-specific BON accuracies and restricted Spearman ρ(Ea, T_pref) for T_pref<1.0.

**Expected finding:** Restricted correlation is stronger (ρ=0.75) than full (ρ=0.67).

In [11]:
def compute_tpref_subgroup(oc_instances: list, mcnemar_result: dict) -> dict:
    logger.info("=== Metric 4: T_pref subgroup analysis ===")
    outcomes = mcnemar_result["per_instance_outcomes"]

    tpref10_idx = [i for i, e in enumerate(oc_instances) if float(e["predict_T_pref"]) == 1.0]
    tpref_lt10_idx = [i for i, e in enumerate(oc_instances) if float(e["predict_T_pref"]) < 1.0]
    n_10 = len(tpref10_idx)
    n_lt10 = len(tpref_lt10_idx)
    frac_10 = n_10 / len(oc_instances)
    logger.info(f"T_pref==1.0: {n_10} ({frac_10:.1%}), T_pref<1.0: {n_lt10}")

    def subgroup_acc(idx, key):
        k = sum(outcomes[i][key] for i in idx)
        n = len(idx)
        acc = k / n if n > 0 else 0
        w_lo, w_hi = wilson_ci(k, n) if n > 0 else (0, 0)
        return {"k": k, "n": n, "accuracy": round(acc, 4), "wilson_lo": round(w_lo, 4), "wilson_hi": round(w_hi, 4)}

    # Subgroup T_pref < 1.0 accuracies
    sub_op03 = subgroup_acc(tpref_lt10_idx, "bon_Toperating_03")
    sub_t10 = subgroup_acc(tpref_lt10_idx, "bon_fixed10")
    sub_t07 = subgroup_acc(tpref_lt10_idx, "bon_fixed07")

    # Spearman ρ(Ea, T_pref) restricted to T_pref < 1.0
    ea_sub = np.array([float(oc_instances[i]["predict_arrhenius_Ea"]) for i in tpref_lt10_idx])
    tpref_sub = np.array([float(oc_instances[i]["predict_T_pref"]) for i in tpref_lt10_idx])
    if len(ea_sub) >= 3:
        rho_sub, p_sub = stats.spearmanr(ea_sub, tpref_sub)
    else:
        rho_sub, p_sub = float("nan"), float("nan")
    logger.info(f"Subgroup T_pref<1.0 Spearman ρ(Ea, T_pref)={rho_sub:.3f}, p={p_sub:.4f}")

    # Also full Ea vs T_pref for the scatter
    ea_all = np.array([float(e["predict_arrhenius_Ea"]) for e in oc_instances])
    tpref_all = np.array([float(e["predict_T_pref"]) for e in oc_instances])
    rho_all, p_all = stats.spearmanr(ea_all, tpref_all)

    logger.info(f"Full Spearman ρ(Ea, T_pref)={rho_all:.3f}, p={p_all:.4f}")

    # T_pref=1.0 subgroup accuracies
    sub10_op03 = subgroup_acc(tpref10_idx, "bon_Toperating_03")
    sub10_t10 = subgroup_acc(tpref10_idx, "bon_fixed10")
    sub10_t07 = subgroup_acc(tpref10_idx, "bon_fixed07")

    return {
        "n_tpref_10": n_10,
        "n_tpref_lt10": n_lt10,
        "frac_tpref_10": round(frac_10, 4),
        "subgroup_lt10": {
            "acc_T_operating": sub_op03,
            "acc_fixed10": sub_t10,
            "acc_fixed07": sub_t07,
            "spearman_Ea_Tpref_restricted": {
                "rho": round(float(rho_sub), 4),
                "p": round(float(p_sub), 4),
                "n": n_lt10,
            },
        },
        "subgroup_10": {
            "acc_T_operating": sub10_op03,
            "acc_fixed10": sub10_t10,
            "acc_fixed07": sub10_t07,
        },
        "full_spearman_Ea_Tpref": {
            "rho": round(float(rho_all), 4),
            "p": round(float(p_all), 6),
        },
        # arrays for plotting
        "_ea_all": ea_all.tolist(),
        "_tpref_all": tpref_all.tolist(),
        "_is_tpref10": [float(e["predict_T_pref"]) == 1.0 for e in oc_instances],
    }


def plot_tpref_subgroup(tpref_result: dict, out_path=None):
    logger.info("Plotting t_tpref10_subgroup_analysis")

    ea_all = np.array(tpref_result["_ea_all"])
    tpref_all = np.array(tpref_result["_tpref_all"])
    is_10 = np.array(tpref_result["_is_tpref10"])
    rho = tpref_result["full_spearman_Ea_Tpref"]["rho"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Panel A: Scatter Ea vs T_pref with jitter
    rng = np.random.default_rng(7)
    jitter = rng.uniform(-0.015, 0.015, size=len(tpref_all))
    ax1.scatter(ea_all[is_10], tpref_all[is_10] + jitter[is_10],
                color="#FF8C00", s=70, alpha=0.8, label=f"T_pref=1.0 (n={is_10.sum()})", zorder=3)
    ax1.scatter(ea_all[~is_10], tpref_all[~is_10] + jitter[~is_10],
                color="#1E90FF", s=70, alpha=0.8, label=f"T_pref<1.0 (n={(~is_10).sum()})", zorder=3)
    ax1.set_xlabel("Ea (Arrhenius activation energy)", fontsize=11)
    ax1.set_ylabel("T_pref (jittered)", fontsize=11)
    ax1.set_title(f"Panel A: Ea vs T_pref (Spearman ρ={rho:.3f})", fontsize=11)
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)
    ax1.text(0.05, 0.95, f"ρ={rho:.3f}", transform=ax1.transAxes,
             fontsize=12, va="top", fontweight="bold")

    # Panel B: Grouped bar chart
    strategies = ["T_operating_0.3", "fixed_T07", "fixed_T10"]
    keys_10 = ["acc_T_operating", "acc_fixed07", "acc_fixed10"]
    keys_lt10 = ["acc_T_operating", "acc_fixed07", "acc_fixed10"]

    sub10 = tpref_result["subgroup_10"]
    sublt10 = tpref_result["subgroup_lt10"]

    acc_10 = [sub10[k]["accuracy"] for k in keys_10]
    acc_lt10 = [sublt10[k]["accuracy"] for k in keys_lt10]
    err_lo_10 = [sub10[k]["accuracy"] - sub10[k]["wilson_lo"] for k in keys_10]
    err_hi_10 = [sub10[k]["wilson_hi"] - sub10[k]["accuracy"] for k in keys_10]
    err_lo_lt10 = [sublt10[k]["accuracy"] - sublt10[k]["wilson_lo"] for k in keys_lt10]
    err_hi_lt10 = [sublt10[k]["wilson_hi"] - sublt10[k]["accuracy"] for k in keys_lt10]

    x = np.arange(len(strategies))
    width = 0.35
    ax2.bar(x - width/2, acc_10, width, label=f"T_pref=1.0 (n={tpref_result['n_tpref_10']})",
            color="#FF8C00", alpha=0.8,
            yerr=[err_lo_10, err_hi_10], capsize=5, error_kw={"ecolor": "#333333"})
    ax2.bar(x + width/2, acc_lt10, width, label=f"T_pref<1.0 (n={tpref_result['n_tpref_lt10']})",
            color="#1E90FF", alpha=0.8,
            yerr=[err_lo_lt10, err_hi_lt10], capsize=5, error_kw={"ecolor": "#333333"})
    ax2.set_xticks(x)
    ax2.set_xticklabels(strategies, fontsize=10)
    ax2.set_ylabel("BON Accuracy (Wilson 95% CI)", fontsize=11)
    ax2.set_title("Panel B: Accuracy by Subgroup\n(T_pref=1.0 vs T_pref<1.0)", fontsize=11)
    ax2.set_ylim(0, 1.15)
    ax2.legend(fontsize=9)
    ax2.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    if out_path:
        fig.savefig(out_path, dpi=150, bbox_inches="tight")
        logger.info(f"Saved {out_path}")
    plt.show()
    plt.close(fig)


tpref_res = compute_tpref_subgroup(oc_instances, mcnemar_res)
plot_tpref_subgroup(tpref_res, out_path="t_tpref10_subgroup_analysis.png")

20:53:31|INFO   |=== Metric 4: T_pref subgroup analysis ===


20:53:31|INFO   |T_pref==1.0: 13 (43.3%), T_pref<1.0: 17


20:53:31|INFO   |Subgroup T_pref<1.0 Spearman ρ(Ea, T_pref)=0.752, p=0.0005


20:53:31|INFO   |Full Spearman ρ(Ea, T_pref)=0.674, p=0.0000


20:53:31|INFO   |Plotting t_tpref10_subgroup_analysis


20:53:31|INFO   |Saved t_tpref10_subgroup_analysis.png


## Metric 5: Power Analysis for Ea–T_pref Correlation

**Reviewer critique (C6):** The Spearman ρ(Ea, T_pref)=0.674 may be an underpowered finding.

**Method:** Fisher z-transform to compute minimum detectable ρ at 80% power (two-sided α=0.05),
and minimum n required to detect the observed ρ at 80% power.

**Expected finding:** C6 is adequately powered — observed ρ >> minimum detectable ρ.

In [12]:
def compute_power_analysis(metadata: dict) -> dict:
    logger.info("=== Metric 5: Power analysis (Ea–T_pref correlation) ===")
    step8 = metadata["step8_Ea_predicts_Tpref"]
    n = step8["n_instances"]
    obs_rho = step8["spearman_Ea_Tpref"]["rho"]

    z_alpha2 = stats.norm.ppf(0.975)  # 1.96
    z_beta = 0.842                      # 80% power

    se_fisher_z = 1.0 / math.sqrt(n - 3)
    min_detectable_z = (z_alpha2 + z_beta) * se_fisher_z
    min_detectable_rho = math.tanh(min_detectable_z)

    # n required to detect observed rho at 80% power
    z_rho = math.atanh(obs_rho)
    n_required = (z_alpha2 + z_beta) ** 2 / z_rho ** 2 + 3

    logger.info(f"n={n}, obs_rho={obs_rho:.4f}, min_detectable_rho={min_detectable_rho:.4f}")
    logger.info(f"n_required for obs_rho at 80% power: {n_required:.1f}")

    interp = (
        f"At n={n}, the minimum detectable ρ with 80% power (two-sided α=0.05) is approximately "
        f"ρ_min≈{min_detectable_rho:.2f}; the observed ρ={obs_rho:.3f} substantially exceeds this, "
        f"confirming adequate power. The C6 finding is not underpowered."
    )
    logger.info(interp)

    return {
        "n": n,
        "observed_rho": round(obs_rho, 6),
        "min_detectable_rho_80pct_power": round(min_detectable_rho, 4),
        "se_fisher_z": round(se_fisher_z, 6),
        "n_required_for_observed_rho_at_80pct": round(n_required, 2),
        "z_alpha_half": round(z_alpha2, 4),
        "z_beta": z_beta,
        "interpretation": interp,
    }


power_res = compute_power_analysis(metadata)

20:53:31|INFO   |=== Metric 5: Power analysis (Ea–T_pref correlation) ===


20:53:31|INFO   |n=30, obs_rho=0.6736, min_detectable_rho=0.4924


20:53:31|INFO   |n_required for obs_rho at 80% power: 14.8


20:53:31|INFO   |At n=30, the minimum detectable ρ with 80% power (two-sided α=0.05) is approximately ρ_min≈0.49; the observed ρ=0.674 substantially exceeds this, confirming adequate power. The C6 finding is not underpowered.


## Metric 6: T_TURN^theory vs T_TURN^emp Reconciliation

**Reviewer critique:** The paper conflates two distinct T_TURN quantities without clarification.

**Method:** Compute T_TURN^theory = Ea/ln(K) for K=10 (MMLU-Pro choices) from each instance's
Arrhenius fit. Compare distribution to T_TURN^emp (Du et al. entropy inflection point, pre-computed
in the parent experiment).

**Expected finding:** T_TURN^theory << T_TURN^emp (ratio ~5×); they validate different claims.

In [13]:
def compute_t_turn_reconciliation(oc_instances: list) -> dict:
    logger.info("=== Metric 6: T_TURN reconciliation ===")
    K = N_CHOICES  # 10
    ln_K = math.log(K)

    ea_vals = np.array([float(e["predict_arrhenius_Ea"]) for e in oc_instances])
    t_turn_emp = np.array([float(e["predict_T_TURN"]) for e in oc_instances])

    t_turn_theory = ea_vals / ln_K

    mean_th = float(np.mean(t_turn_theory))
    median_th = float(np.median(t_turn_theory))
    iqr_th_lo = float(np.percentile(t_turn_theory, 25))
    iqr_th_hi = float(np.percentile(t_turn_theory, 75))

    mean_emp = float(np.mean(t_turn_emp))
    median_emp = float(np.median(t_turn_emp))
    iqr_emp_lo = float(np.percentile(t_turn_emp, 25))
    iqr_emp_hi = float(np.percentile(t_turn_emp, 75))

    frac_theory_lt_emp = float(np.mean(t_turn_theory < t_turn_emp))

    logger.info(f"T_TURN^theory: mean={mean_th:.3f}, median={median_th:.3f}, IQR=[{iqr_th_lo:.3f},{iqr_th_hi:.3f}]")
    logger.info(f"T_TURN^emp:   mean={mean_emp:.3f}, median={median_emp:.3f}, IQR=[{iqr_emp_lo:.3f},{iqr_emp_hi:.3f}]")
    logger.info(f"Fraction T_TURN^theory < T_TURN^emp: {frac_theory_lt_emp:.3f}")

    clarification = (
        f"T_TURN^theory = Ea/ln(K) (mean≈{mean_th:.2f}, median≈{median_th:.2f}, "
        f"IQR≈[{iqr_th_lo:.2f},{iqr_th_hi:.2f}]) is the theoretical upper bound quantity in Theorem 6, "
        f"proving window [T_thresh, T_TURN^theory] is non-empty when N>K; it is distinct from "
        f"T_TURN^emp = Du et al. entropy inflection point "
        f"(mean≈{mean_emp:.2f}, median≈{median_emp:.2f}, IQR≈[{iqr_emp_lo:.2f},{iqr_emp_hi:.2f}]). "
        f"The 93.3% window fraction uses T_TURN^emp; Theorem 6 proves a complementary mathematical "
        f"bound using T_TURN^theory. These quantities are not interchangeable and validate different "
        f"aspects of the framework."
    )

    return {
        "K": K,
        "ln_K": round(ln_K, 6),
        "mean_theory": round(mean_th, 6),
        "median_theory": round(median_th, 6),
        "iqr_theory": [round(iqr_th_lo, 4), round(iqr_th_hi, 4)],
        "mean_emp": round(mean_emp, 6),
        "median_emp": round(median_emp, 6),
        "iqr_emp": [round(iqr_emp_lo, 4), round(iqr_emp_hi, 4)],
        "frac_theory_lt_emp": round(frac_theory_lt_emp, 4),
        "clarification": clarification,
    }


t_turn_res = compute_t_turn_reconciliation(oc_instances)

20:53:31|INFO   |=== Metric 6: T_TURN reconciliation ===


20:53:31|INFO   |T_TURN^theory: mean=0.242, median=0.153, IQR=[0.069,0.304]


20:53:31|INFO   |T_TURN^emp:   mean=1.217, median=1.400, IQR=[1.325,1.400]


20:53:31|INFO   |Fraction T_TURN^theory < T_TURN^emp: 0.933


## Metric 7: Delta-Approx Methodology Note

**Reviewer critique:** The comparison between regression-Ea and delta-approx strategies
conflates accuracy differences with information cost differences.

**Finding:** Regression-Ea uses ~350 API calls/instance while delta-approx uses 1 logprob call.
The 3.3 pp accuracy gap is confounded with information cost.

In [14]:
def compute_delta_approx_note(metadata: dict) -> dict:
    logger.info("=== Metric 7: Delta-approx methodology note ===")
    acc_map = metadata["step7_accuracy_comparison"]["accuracy"]
    # Regression Ea → T_operating_delta_0.3 at 90.0%
    acc_regression = acc_map["T_operating_delta_0.3"]
    # Delta-approx is the single logprob approach; from step5 it uses Δ=logit(wrong)-logit(correct)
    # The plan says delta-approx achieves 85.8% — this was noted in the hypothesis as T_operating_delta_0.2
    acc_delta_approx = acc_map["T_operating_delta_0.2"]
    gap_pp = round((acc_regression - acc_delta_approx) * 100, 2)

    note = (
        f"The delta-approx strategy (T_operating from single logprob call, "
        f"accuracy={acc_delta_approx:.1%}) achieves {gap_pp:.1f} pp less than "
        f"regression-Ea T_operating (accuracy={acc_regression:.1%}). "
        f"These are compared via a methodology asymmetry: regression Ea uses ~350 API calls "
        f"per instance (7 temperatures × 50 samples) while delta-approx uses 1 logprob call. "
        f"The accuracy difference is therefore confounded with information cost."
    )

    return {
        "accuracy_regression_ea": round(acc_regression, 6),
        "accuracy_delta_approx": round(acc_delta_approx, 6),
        "gap_pp": gap_pp,
        "api_calls_regression": 350,
        "api_calls_delta_approx": 1,
        "methodological_note": note,
    }


delta_note = compute_delta_approx_note(metadata)

20:53:31|INFO   |=== Metric 7: Delta-approx methodology note ===


## Verdict Updates

Based on the six metrics, three criteria receive updated verdicts:
- **C1 (median R²):** INDETERMINATE — threshold 0.85 falls within the bootstrap CI
- **C7 (T_operating vs fixed):** NOT SIGNIFICANT — McNemar p > 0.05 at n=30
- **C6 (Ea–T_pref power):** CONFIRMED — observed ρ substantially exceeds minimum detectable ρ

In [15]:
def compute_verdict_updates(mcnemar_res: dict, bootstrap_res: dict, power_res: dict) -> dict:
    mc_pval = float(mcnemar_res["mcnemar_Toperating_vs_fixed07"]["pvalue"])
    c1_in_ci = bootstrap_res["threshold_085_in_ci"]

    return {
        "C1_median_R2_verdict": (
            "INDETERMINATE" if c1_in_ci else "NOT_MET"
        ),
        "C1_rationale": (
            f"Observed median R²={bootstrap_res['median']:.3f}, 95% bootstrap CI "
            f"[{bootstrap_res['ci_lo']:.3f}, {bootstrap_res['ci_hi']:.3f}]. "
            f"Threshold 0.85 {'falls within' if c1_in_ci else 'falls outside'} CI → "
            f"verdict is {'INDETERMINATE' if c1_in_ci else 'NOT_MET'}."
        ),
        "C7_verdict_downgrade": mc_pval > 0.05,
        "C7_rationale": (
            f"McNemar p={mc_pval:.3f} (n=30); difference not statistically significant. "
            f"Directionally consistent but confirmation requires n≈200."
        ),
        "C6_power_confirmed": power_res["observed_rho"] > power_res["min_detectable_rho_80pct_power"],
        "C6_rationale": power_res["interpretation"],
    }


verdict_updates = compute_verdict_updates(mcnemar_res, bootstrap_res, power_res)

## Results Summary

Consolidated view of all six metrics and verdict updates.

In [16]:
import textwrap

print("=" * 70)
print("STATISTICAL RIGOR EVALUATION — SUMMARY")
print("=" * 70)

print("\n[1] McNemar's Exact Test")
m1 = mcnemar_res["mcnemar_Toperating_vs_fixed07"]
m2 = mcnemar_res["mcnemar_Toperating_vs_fixed10"]
print(f"  T_operating vs fixed_T07: table={m1['contingency_table']}, p={m1['pvalue']:.4f}")
print(f"  T_operating vs fixed_T10: table={m2['contingency_table']}, p={m2['pvalue']:.4f}")
print(f"  → Neither reaches significance (need n≈200 for 80% power)")

print("\n[2] Table 2 Clopper-Pearson CIs")
n_ovlp = table2_ci_agg["n_strategies_ci_overlap_with_top"]
print(f"  {n_ovlp}/5 non-reference strategies overlap CIs with T_operating_0.3")
for r in sorted(table2_cis, key=lambda x: -x["accuracy"]):
    print(f"  {r['strategy']:30s}: {r['accuracy']:.3f} CP=[{r['cp_lo']:.3f}, {r['cp_hi']:.3f}]")

print("\n[3] Bootstrap CI on Median R²")
print(f"  Observed median R² = {bootstrap_res['median']:.4f}")
print(f"  95% CI (B={B}) = [{bootstrap_res['ci_lo']:.4f}, {bootstrap_res['ci_hi']:.4f}]")
print(f"  C1 threshold 0.85 in CI: {bootstrap_res['threshold_085_in_ci']} → {verdict_updates['C1_median_R2_verdict']}")

print("\n[4] T_pref Subgroup Analysis")
print(f"  T_pref=1.0: n={tpref_res['n_tpref_10']} ({tpref_res['frac_tpref_10']:.1%})")
print(f"  T_pref<1.0: n={tpref_res['n_tpref_lt10']}")
r = tpref_res["subgroup_lt10"]["spearman_Ea_Tpref_restricted"]
print(f"  Restricted ρ(Ea, T_pref) = {r['rho']:.4f} (p={r['p']:.4f}) — stronger than full ρ={tpref_res['full_spearman_Ea_Tpref']['rho']:.4f}")

print("\n[5] Power Analysis (C6)")
print(f"  Observed ρ = {power_res['observed_rho']:.4f}")
print(f"  Min detectable ρ (80% power) = {power_res['min_detectable_rho_80pct_power']:.4f}")
print(f"  n required to detect observed ρ = {power_res['n_required_for_observed_rho_at_80pct']:.1f}")
print(f"  C6 power confirmed: {verdict_updates['C6_power_confirmed']}")

print("\n[6] T_TURN Reconciliation")
print(f"  T_TURN^theory (Ea/ln(K)): mean={t_turn_res['mean_theory']:.3f}, median={t_turn_res['median_theory']:.3f}")
print(f"  T_TURN^emp (Du et al.):   mean={t_turn_res['mean_emp']:.3f}, median={t_turn_res['median_emp']:.3f}")
print(f"  Ratio theory/emp: {t_turn_res['mean_theory']/t_turn_res['mean_emp']:.2f}x")
print(f"  Fraction theory < emp: {t_turn_res['frac_theory_lt_emp']:.1%}")

print("\n[7] Delta-Approx Methodology Note")
print(f"  Regression-Ea accuracy: {delta_note['accuracy_regression_ea']:.3f} ({delta_note['api_calls_regression']} API calls)")
print(f"  Delta-approx accuracy:  {delta_note['accuracy_delta_approx']:.3f} ({delta_note['api_calls_delta_approx']} API call)")
print(f"  Gap: {delta_note['gap_pp']:.1f} pp (confounded with information cost)")

print("\n" + "=" * 70)
print("VERDICT UPDATES")
print("=" * 70)
print(f"  C1 (median R²):     {verdict_updates['C1_median_R2_verdict']}")
print(f"  C7 (T_op vs fixed): {'DOWNGRADED (not significant)' if verdict_updates['C7_verdict_downgrade'] else 'MAINTAINED'}")
print(f"  C6 (power):         {'CONFIRMED' if verdict_updates['C6_power_confirmed'] else 'UNCONFIRMED'}")

# Visualization: bootstrap distribution of median R²
rng = np.random.default_rng(BOOTSTRAP_SEED)
r2_vals = np.array([float(e["predict_arrhenius_R2"]) for e in oc_instances])
boot_medians = np.array([np.median(rng.choice(r2_vals, size=len(r2_vals), replace=True)) for _ in range(B)])

matplotlib.use("Agg")
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: Bootstrap distribution
axes[0].hist(boot_medians, bins=20, color="steelblue", alpha=0.75, edgecolor="white")
axes[0].axvline(bootstrap_res["median"], color="navy", linewidth=2, label=f"Observed median={bootstrap_res['median']:.3f}")
axes[0].axvline(0.85, color="red", linewidth=2, linestyle="--", label="C1 threshold=0.85")
axes[0].axvline(bootstrap_res["ci_lo"], color="orange", linewidth=1.5, linestyle=":")
axes[0].axvline(bootstrap_res["ci_hi"], color="orange", linewidth=1.5, linestyle=":", label=f"95% CI [{bootstrap_res['ci_lo']:.3f}, {bootstrap_res['ci_hi']:.3f}]")
axes[0].set_xlabel("Bootstrap Median R²", fontsize=11)
axes[0].set_ylabel("Count", fontsize=11)
axes[0].set_title(f"Bootstrap CI on Median R² (B={B})", fontsize=11)
axes[0].legend(fontsize=8)

# Right: Verdict summary bar chart
verdicts = {
    "C1: Median R²": 0 if verdict_updates["C1_median_R2_verdict"] == "INDETERMINATE" else 1,
    "C6: Power": 1 if verdict_updates["C6_power_confirmed"] else 0,
    "C7: T_op beats fixed": 0 if verdict_updates["C7_verdict_downgrade"] else 1,
}
colors = ["orange", "green", "red"]
labels = list(verdicts.keys())
values = ["INDETERM.", "CONFIRMED", "DOWNGRADED"]
y_pos = range(len(labels))
axes[1].barh(list(y_pos), [1, 1, 1], color=colors, alpha=0.7)
axes[1].set_yticks(list(y_pos))
axes[1].set_yticklabels(labels, fontsize=10)
for i, v in enumerate(values):
    axes[1].text(0.5, i, v, ha="center", va="center", fontsize=11, fontweight="bold", color="white")
axes[1].set_xlim(0, 1)
axes[1].set_xticks([])
axes[1].set_title("Verdict Updates", fontsize=11)

plt.tight_layout()
plt.savefig("summary_verdicts.png", dpi=150, bbox_inches="tight")
plt.show()
print("\nSummary plot saved to summary_verdicts.png")

STATISTICAL RIGOR EVALUATION — SUMMARY

[1] McNemar's Exact Test
  T_operating vs fixed_T07: table=[24, 2, 2, 2], p=1.0000
  T_operating vs fixed_T10: table=[26, 0, 2, 2], p=0.5000
  → Neither reaches significance (need n≈200 for 80% power)

[2] Table 2 Clopper-Pearson CIs
  5/5 non-reference strategies overlap CIs with T_operating_0.3
  TURN_adapted                  : 0.967 CP=[0.828, 0.999]
  fixed_T10                     : 0.933 CP=[0.779, 0.992]
  T_operating_delta_0.3         : 0.900 CP=[0.735, 0.979]
  T_operating_delta_0.2         : 0.867 CP=[0.693, 0.962]
  fixed_T07                     : 0.833 CP=[0.653, 0.944]
  T_operating_delta_0.4         : 0.800 CP=[0.614, 0.923]

[3] Bootstrap CI on Median R²
  Observed median R² = 0.8477
  95% CI (B=100) = [0.7705, 0.9111]
  C1 threshold 0.85 in CI: True → INDETERMINATE

[4] T_pref Subgroup Analysis
  T_pref=1.0: n=13 (43.3%)
  T_pref<1.0: n=17
  Restricted ρ(Ea, T_pref) = 0.7518 (p=0.0005) — stronger than full ρ=0.6736

[5] Power Analy


Summary plot saved to summary_verdicts.png
